# SwiGLU Calibration and Recovery Study

This notebook continues the SwiGLU allocation study with a calibration-data sweep,
larger model-wide KL recovery budgets, and a global sparsity sweep.
It uses the winning configuration from swiglu-2.

The experiments run in the Python workflow. This notebook only loads the saved
JSON results and reproduces their tables, plots, and interpretation.
Model weights, training data, and a GPU are not required.

setup

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')


PROJECT_ROOT = find_project_root(Path.cwd())
SWIGLU_3_ARTIFACT_PATH = (
    PROJECT_ROOT / 'data/results/workflows/models/swiglu-3/run-001.json'
)
REFERENCE_ARTIFACT_PATH = (
    PROJECT_ROOT / 'data/results/notebook-model-study/swiglu-2.json'
)

In [ ]:
swiglu_3_artifact = json.loads(
    SWIGLU_3_ARTIFACT_PATH.read_text(encoding='utf-8')
)
reference_artifact = json.loads(
    REFERENCE_ARTIFACT_PATH.read_text(encoding='utf-8')
)
if (
    swiglu_3_artifact['schema_version'] != 1
    or swiglu_3_artifact['workflow'] != 'swiglu-3'
    or reference_artifact['schema_version'] != 1
):
    raise ValueError('Incompatible experiment artifact')
if (
    swiglu_3_artifact['status'] != 'completed'
    or swiglu_3_artifact['configuration']['execution_mode'] != 'scientific'
):
    raise ValueError('Use a completed scientific run for this report')
reference_sha256 = hashlib.sha256(REFERENCE_ARTIFACT_PATH.read_bytes()).hexdigest()
if reference_sha256 != swiglu_3_artifact['provenance']['source_sha256']['allocation_artifact']:
    raise ValueError('The swiglu-2 reference differs from the recorded source')

configuration = swiglu_3_artifact['configuration']
results = swiglu_3_artifact['results']
for stage in ('calibration', 'sparsity', 'recovery'):
    if not results[stage]['completed']:
        raise ValueError(f'Incomplete stage: {stage}')
trajectories = results['recovery']['trajectories']
if any(row['status'] != 'completed' for row in trajectories.values()):
    raise ValueError('A recovery trajectory is incomplete')

dense_reference_df = pd.DataFrame(reference_artifact['results']['dense_reference'])
dense_reference = dense_reference_df.iloc[0]
SELECTED_CALIBRATION_PAIRS = results['calibration']['selected_calibration_pairs']
SPARSITY_KEYS = sorted(trajectories, key=float)
SPARSITY_LABELS = {key: f'{float(key):.0%}' for key in SPARSITY_KEYS}
SPARSITY_COLORS = dict(zip(
    SPARSITY_LABELS.values(), sns.color_palette('deep', len(SPARSITY_KEYS))
))

In [ ]:
configuration_df = pd.DataFrame([
    ('Model', configuration['model']['model_id']),
    ('Revision', configuration['model']['revision']),
    ('Initialization', configuration['local_fitting']['initialization']),
    ('Allocation score', configuration['allocation']['score']),
    ('Allocation temperature', configuration['allocation']['temperature']),
    ('Retention floor', configuration['allocation']['minimum_retention']),
    ('Protected layers', configuration['allocation']['protected_layers']),
    ('Calibration pairs', configuration['calibration']['pair_counts']),
    ('Capture group size used', configuration['calibration']['capture_group_size']),
    ('Recovery tokens per model', configuration['recovery']['target_tokens_per_model']),
    ('Recovery learning rate', configuration['recovery']['learning_rate']),
    ('Effective batch tokens', results['recovery']['effective_batch_tokens']),
    ('Recovery parameter dtype', configuration['recovery']['replacement_parameter_dtype']),
    ('Recovery forward dtype', configuration['recovery']['forward_autocast_dtype']),
    ('Sequence length', configuration['data']['sequence_length']),
], columns=['setting', 'value'])
display(configuration_df)
display(dense_reference_df)
print(f'Loaded: {SWIGLU_3_ARTIFACT_PATH.relative_to(PROJECT_ROOT)}')

#### Calibration data sweep

In [ ]:
calibration_fitting_df = pd.DataFrame(results['calibration']['operator_fitting'])
calibration_history_df = pd.DataFrame(results['calibration']['operator_training_history'])
calibration_model_df = pd.json_normalize(results['calibration']['model_evaluation'])
calibration_model_df = calibration_model_df.rename(columns={
    'allocation_selection.teacher_kl': 'c4_selection_kl',
    'allocation_selection.loss': 'c4_selection_loss',
    'allocation_selection.perplexity': 'c4_selection_perplexity',
    'wikitext_validation.loss': 'wikitext_loss',
    'wikitext_validation.perplexity': 'wikitext_perplexity',
})
calibration_summary_df = (
    calibration_fitting_df.groupby('calibration_pairs', as_index=False)
    .agg(
        blocks=('layer', 'size'),
        mean_nmse=('local_nmse', 'mean'),
        median_nmse=('local_nmse', 'median'),
        worst_nmse=('local_nmse', 'max'),
        mean_cosine=('local_cosine', 'mean'),
        mean_best_epoch=('best_epoch', 'mean'),
        mean_epochs=('epochs_completed', 'mean'),
        mean_updates=('updates', 'mean'),
        total_fit_minutes=('fit_seconds', lambda values: values.sum() / 60),
    )
    .merge(calibration_model_df[[
        'calibration_pairs', 'c4_selection_kl',
        'c4_selection_perplexity', 'wikitext_perplexity',
    ]], on='calibration_pairs', validate='one_to_one')
    .sort_values('calibration_pairs').reset_index(drop=True)
)
calibration_summary_df['selected'] = (
    calibration_summary_df['calibration_pairs'] == SELECTED_CALIBRATION_PAIRS
)

reporting

In [ ]:
display(calibration_summary_df[[
    'calibration_pairs', 'blocks', 'mean_nmse', 'median_nmse', 'worst_nmse',
    'mean_cosine', 'c4_selection_kl', 'c4_selection_perplexity',
    'wikitext_perplexity', 'selected',
]])
display(calibration_summary_df[[
    'calibration_pairs', 'mean_best_epoch', 'mean_epochs',
    'mean_updates', 'total_fit_minutes',
]])
print(f'Selected calibration budget: {SELECTED_CALIBRATION_PAIRS:,} pairs per block')

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, metric, label in zip(
    axes,
    ('c4_selection_kl', 'wikitext_perplexity'),
    ('C4 allocation-selection KL', 'WikiText validation perplexity'),
):
    sns.lineplot(
        data=calibration_summary_df, x='calibration_pairs', y=metric,
        marker='o', estimator=None, ax=axis,
    )
    selected_row = calibration_summary_df.query('selected').iloc[0]
    axis.scatter(
        [SELECTED_CALIBRATION_PAIRS], [selected_row[metric]],
        marker='*', s=180, color='black', label='Selected budget', zorder=3,
    )
    axis.set_xticks(calibration_summary_df['calibration_pairs'])
    axis.set_xticklabels([
        f'{value:,}' for value in calibration_summary_df['calibration_pairs']
    ])
    axis.set(xlabel='Calibration pairs per block', ylabel=label)
    axis.legend()
figure.suptitle('Calibration budget at fixed 50% eligible-MLP sparsity')
figure.tight_layout()
plt.show()

In [ ]:
baseline_pairs = int(calibration_summary_df['calibration_pairs'].min())
calibration_block_df = calibration_fitting_df.merge(
    calibration_fitting_df.query('calibration_pairs == @baseline_pairs')
    [['layer', 'local_nmse']].rename(columns={'local_nmse': 'baseline_nmse'}),
    on='layer', validate='many_to_one',
)
calibration_block_df['nmse_improvement_pct'] = 100 * (
    1 - calibration_block_df['local_nmse'] / calibration_block_df['baseline_nmse']
)
calibration_block_df['budget'] = calibration_block_df['calibration_pairs'].map(
    lambda value: f'{value:,}'
)
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(
    data=calibration_block_df, x='layer', y='local_nmse', hue='budget',
    marker='o', estimator=None, ax=axes[0],
)
axes[0].set(yscale='log', xlabel='Transformer block', ylabel='Local validation NMSE (log scale)')
sns.lineplot(
    data=calibration_block_df.query('calibration_pairs > @baseline_pairs'),
    x='layer', y='nmse_improvement_pct', hue='budget',
    marker='o', estimator=None, ax=axes[1],
)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set(xlabel='Transformer block', ylabel=f'NMSE reduction from {baseline_pairs:,} pairs (%)')
figure.suptitle('Local fitting gains by block')
figure.tight_layout()
plt.show()

In [ ]:
DETAIL_LAYERS = [2, 13, 22]
figure, axes = plt.subplots(1, len(DETAIL_LAYERS), figsize=(15, 4), squeeze=False)
for axis, layer in zip(axes.flat, DETAIL_LAYERS):
    layer_history = calibration_history_df.query('layer == @layer')
    for pairs, history in layer_history.groupby('calibration_pairs', sort=True):
        history = history.sort_values('updates')
        line, = axis.plot(
            history['updates'], history['validation_nmse'], label=f'{pairs:,} pairs'
        )
        fit = calibration_fitting_df.query(
            'layer == @layer and calibration_pairs == @pairs'
        ).iloc[0]
        if fit['best_epoch'] == 0:
            axis.scatter(0, fit['initial_local_nmse'], marker='*', s=100, color=line.get_color())
        else:
            best = history.loc[history['epoch'] == fit['best_epoch']].iloc[0]
            axis.scatter(best['updates'], best['validation_nmse'], marker='*', s=100, color=line.get_color())
    axis.set(title=f'Block {layer}', xlabel='Completed optimizer updates', ylabel='Validation NMSE')
    axis.legend(fontsize='small')
figure.suptitle('Local fitting convergence')
figure.tight_layout()
plt.show()

#### Model retraining full budget

In [ ]:
pre_recovery_df = pd.json_normalize(results['sparsity']['model_evaluation'])
history_rows = []
milestone_rows = []
resource_rows = []
for key in SPARSITY_KEYS:
    trajectory = trajectories[key]
    history_rows.extend({'sparsity_key': key, **row} for row in trajectory['validation_history'])
    pre = next(row for row in results['sparsity']['model_evaluation'] if row['sparsity_key'] == key)
    milestone_rows.append({
        'sparsity_key': key, 'checkpoint': 'Before recovery',
        'requested_tokens': 0, 'tokens_seen': 0,
        'recovery_validation_kl': trajectory['validation_history'][0]['recovery_validation_kl'],
        'allocation_selection': pre['allocation_selection'],
        'wikitext_validation': pre['wikitext_validation'],
    })
    for milestone in trajectory['milestones']:
        for requested in milestone['requested_tokens']:
            for state in ('current', 'best_under_budget'):
                evaluation = milestone[state]
                milestone_rows.append({
                    'sparsity_key': key,
                    'checkpoint': f'{requested / 1e6:g}M {state}',
                    'requested_tokens': requested,
                    'tokens_seen': (milestone['actual_tokens'] if state == 'current'
                                    else evaluation['checkpoint_tokens']),
                    **evaluation,
                })
    resource_rows.append({
        'sparsity_key': key,
        'tokens': trajectory['tokens_seen'],
        'updates': trajectory['optimizer_updates'],
        'recovery_hours': trajectory['elapsed_seconds'] / 3600,
        'tokens_per_second': trajectory['tokens_seen'] / trajectory['elapsed_seconds'],
        **trajectory['memory'],
    })

recovery_history_df = pd.DataFrame(history_rows).sort_values(['sparsity_key', 'tokens_seen'])
recovery_history_df['sparsity'] = recovery_history_df['sparsity_key'].map(SPARSITY_LABELS)
recovery_history_df['tokens_millions'] = recovery_history_df['tokens_seen'] / 1e6
recovery_history_df['initial_kl'] = recovery_history_df.groupby('sparsity_key')['recovery_validation_kl'].transform('first')
recovery_history_df['kl_reduction_pct'] = 100 * (
    1 - recovery_history_df['recovery_validation_kl'] / recovery_history_df['initial_kl']
)
recovery_milestone_df = pd.json_normalize(milestone_rows).rename(columns={
    'allocation_selection.teacher_kl': 'c4_selection_kl',
    'allocation_selection.perplexity': 'c4_selection_perplexity',
    'wikitext_validation.loss': 'wikitext_loss',
    'wikitext_validation.perplexity': 'wikitext_perplexity',
})
resource_df = pd.DataFrame(resource_rows)

reporting

In [ ]:
display(recovery_milestone_df[[
    'sparsity_key', 'checkpoint', 'tokens_seen', 'recovery_validation_kl',
    'c4_selection_kl', 'c4_selection_perplexity', 'wikitext_perplexity',
]])

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for axis, metric, label in zip(
    axes,
    ('recovery_validation_kl', 'kl_reduction_pct'),
    ('C4 recovery-validation KL', 'KL reduction from pre-recovery (%)'),
):
    sns.lineplot(
        data=recovery_history_df, x='tokens_millions', y=metric,
        hue='sparsity', hue_order=list(SPARSITY_LABELS.values()),
        palette=SPARSITY_COLORS, estimator=None, ax=axis,
    )
    for key in SPARSITY_KEYS:
        best_tokens = trajectories[key]['selected_checkpoint']['tokens_seen']
        best_row = recovery_history_df.query(
            'sparsity_key == @key and tokens_seen == @best_tokens'
        ).iloc[0]
        axis.scatter(best_tokens / 1e6, best_row[metric], marker='*', s=110,
                     color=SPARSITY_COLORS[SPARSITY_LABELS[key]], zorder=3)
    for tokens in configuration['recovery']['milestone_tokens']:
        axis.axvline(tokens / 1e6, color='grey', linestyle='--', linewidth=1)
    axis.set(xlabel='Recovery tokens (millions)', ylabel=label)
    axis.legend(title='Eligible-MLP sparsity')
figure.suptitle('Recovery trajectories (stars: selected checkpoints)')
figure.tight_layout()
plt.show()

In [ ]:
CHECKPOINT_TOKENS = [0, 10_000_000, 30_000_000, 50_000_000, 80_000_000, 100_000_000]
checkpoint_rows = []
for key in SPARSITY_KEYS:
    history = trajectories[key]['validation_history']
    for requested in CHECKPOINT_TOKENS:
        matches = [row for row in history if (
            (requested == 0 and row['tokens_seen'] == 0)
            or requested in row.get('requested_checkpoint_tokens', [])
        )]
        if len(matches) != 1:
            raise ValueError(f'Expected one saved checkpoint for {key}, {requested}')
        row = matches[0]
        checkpoint_rows.append({
            'sparsity': SPARSITY_LABELS[key],
            'requested_millions': requested / 1e6,
            'actual_tokens': row['tokens_seen'],
            'recovery_validation_kl': row['recovery_validation_kl'],
        })
checkpoint_df = pd.DataFrame(checkpoint_rows)
display(checkpoint_df.pivot(
    index='requested_millions', columns='sparsity', values='recovery_validation_kl'
))
print('Requested budgets are rounded to completed optimizer updates; actual counts are in checkpoint_df.')

In [ ]:
recovery_gain_rows = []
for key in SPARSITY_KEYS:
    rows = recovery_milestone_df.query('sparsity_key == @key')
    before = rows.loc[rows['checkpoint'] == 'Before recovery'].iloc[0]
    first = rows.loc[rows['checkpoint'] == '10M current'].iloc[0]
    selected = trajectories[key]['post_recovery_model_evaluation']
    selected_loss = selected['wikitext_validation']['loss']
    recovered_loss = before['wikitext_loss'] - selected_loss
    recovery_gain_rows.append({
        'sparsity_key': key,
        'selected_tokens': trajectories[key]['selected_checkpoint']['tokens_seen'],
        'recovery_kl_reduction_pct': 100 * (1 - selected['recovery_validation_kl'] / before['recovery_validation_kl']),
        'wikitext_loss_gap_recovered_pct': 100 * recovered_loss / (before['wikitext_loss'] - dense_reference['loss']),
        'share_of_loss_improvement_at_10m_pct': (
            100 * (before['wikitext_loss'] - first['wikitext_loss']) / recovered_loss
            if recovered_loss != 0 else float('nan')
        ),
    })
recovery_gain_df = pd.DataFrame(recovery_gain_rows)
display(recovery_gain_df)
display(resource_df)

#### Global sparsity sweep

In [ ]:
allocation_df = pd.DataFrame(results['sparsity']['allocation'])
allocation_summary_df = pd.DataFrame(results['sparsity']['allocation_summary'])
selected_calibration_fitting_df = calibration_fitting_df.query(
    'calibration_pairs == @SELECTED_CALIBRATION_PAIRS'
).assign(sparsity_key='0.5')
sparsity_fitting_df = pd.concat([
    selected_calibration_fitting_df,
    pd.DataFrame(results['sparsity']['operator_fitting']),
], ignore_index=True)
local_sparsity_summary_df = (
    sparsity_fitting_df.groupby('sparsity_key', as_index=False)
    .agg(blocks=('layer', 'size'), mean_nmse=('local_nmse', 'mean'),
         worst_nmse=('local_nmse', 'max'), mean_cosine=('local_cosine', 'mean'))
)

quality_rows = []
for key in SPARSITY_KEYS:
    trajectory = trajectories[key]
    pre = pre_recovery_df.loc[pre_recovery_df['sparsity_key'] == key].iloc[0]
    milestone_10 = next(row for row in trajectory['milestones'] if 10_000_000 in row['requested_tokens'])
    milestone_100 = next(row for row in trajectory['milestones'] if 100_000_000 in row['requested_tokens'])
    selected = trajectory['post_recovery_model_evaluation']
    quality_rows.append({
        'sparsity_key': key,
        'pre_perplexity': pre['wikitext_validation.perplexity'],
        'perplexity_10m': milestone_10['current']['wikitext_validation']['perplexity'],
        'perplexity_100m': milestone_100['current']['wikitext_validation']['perplexity'],
        'selected_perplexity': selected['wikitext_validation']['perplexity'],
        'selected_c4_kl': selected['allocation_selection']['teacher_kl'],
    })
quality_summary_df = (
    allocation_summary_df.merge(pd.DataFrame(quality_rows), on='sparsity_key', validate='one_to_one')
    .merge(resource_df[['sparsity_key', 'recovery_hours']], on='sparsity_key', validate='one_to_one')
    .merge(recovery_gain_df, on='sparsity_key', validate='one_to_one')
    .sort_values('requested_eligible_mlp_removal').reset_index(drop=True)
)
quality_summary_df['model_parameter_reduction_pct'] = 100 * quality_summary_df['realized_whole_model_removal']
quality_summary_df['remaining_parameters'] = int(dense_reference['parameters']) - quality_summary_df['realized_removed_parameters']
quality_summary_df['selected_ppl_increase_pct'] = 100 * (
    quality_summary_df['selected_perplexity'] / dense_reference['perplexity'] - 1
)

reporting

In [ ]:
display(quality_summary_df[[
    'sparsity_key', 'realized_eligible_mlp_removal', 'model_parameter_reduction_pct',
    'remaining_parameters', 'minimum_retention', 'maximum_retention',
    'pre_perplexity', 'perplexity_10m', 'perplexity_100m',
    'selected_perplexity', 'selected_ppl_increase_pct',
]])
display(local_sparsity_summary_df)

In [ ]:
quality_plot_df = quality_summary_df.melt(
    id_vars='model_parameter_reduction_pct',
    value_vars=['pre_perplexity', 'perplexity_10m', 'selected_perplexity'],
    var_name='phase', value_name='perplexity',
)
quality_plot_df['phase'] = quality_plot_df['phase'].map({
    'pre_perplexity': 'Before recovery',
    'perplexity_10m': '10M current',
    'selected_perplexity': 'Selected within 100M',
})
figure, axis = plt.subplots(figsize=(9, 5))
sns.lineplot(
    data=quality_plot_df, x='model_parameter_reduction_pct', y='perplexity',
    hue='phase', style='phase', marker='o', estimator=None, ax=axis,
)
axis.scatter(0, dense_reference['perplexity'], color='black', marker='D', s=65,
             label='Dense reference', zorder=3)
axis.axhline(dense_reference['perplexity'], color='grey', linestyle='--', linewidth=1)
axis.set(xlabel='Whole-model parameter reduction (%)', ylabel='WikiText validation perplexity',
         title='Compression and recovery trade-off')
axis.legend(title='Model state')
figure.tight_layout()
plt.show()

In [ ]:
retention_df = allocation_df.pivot(
    index='sparsity_key', columns='layer', values='replacement_width_ratio'
).reindex(index=SPARSITY_KEYS, columns=range(configuration['model']['num_layers']))
for layer in configuration['allocation']['protected_layers']:
    retention_df[layer] = 1.0
retention_df.index = [SPARSITY_LABELS[key] for key in retention_df.index]
figure, axis = plt.subplots(figsize=(15, 3.5))
sns.heatmap(
    retention_df, vmin=0, vmax=1, cmap='viridis', annot=True, fmt='.2f',
    annot_kws={'fontsize': 8}, cbar_kws={'label': 'Retained SwiGLU width ratio'}, ax=axis,
)
axis.set_xticklabels([
    f'{layer}*' if layer in configuration['allocation']['protected_layers'] else str(layer)
    for layer in retention_df.columns
], rotation=0)
axis.set(xlabel='Transformer block (* protected)', ylabel='Eligible-MLP sparsity',
         title='Layer widths under the fixed allocation policy')
figure.tight_layout()
plt.show()

#### Workflow Summary

In [ ]:
final_summary_df = quality_summary_df[[
    'sparsity_key', 'model_parameter_reduction_pct', 'remaining_parameters',
    'pre_perplexity', 'perplexity_10m', 'perplexity_100m',
    'selected_perplexity', 'selected_tokens', 'recovery_hours',
]].copy()
display(final_summary_df.round(3))

runtime_df = pd.DataFrame(results['runtime'])
runtime_df['hours'] = runtime_df['seconds'] / 3600
display(runtime_df[['stage', 'hours']])
workflow_hours = (
    pd.Timestamp(swiglu_3_artifact['completed_at_utc'])
    - pd.Timestamp(swiglu_3_artifact['created_at_utc'])
).total_seconds() / 3600
print(f'Total workflow wall time: {workflow_hours:.2f} hours')
print(f'Total recovery tokens across models: {results["recovery"]["total_recovery_tokens"]:,}')

In [ ]:
winning_policy = configuration['references']['winning_policy']
historical_winner = next(
    row for row in reference_artifact['results']['final_summary']
    if row['policy'] == winning_policy
)
current_50 = quality_summary_df.loc[quality_summary_df['sparsity_key'] == '0.5'].iloc[0]
historical_comparison_df = pd.DataFrame([
    {'experiment': 'swiglu-2 winner',
     'pre_perplexity': historical_winner['pre_perplexity'],
     'post_perplexity': historical_winner['post_perplexity']},
    {'experiment': 'swiglu-3 selected within 100M',
     'pre_perplexity': current_50['pre_perplexity'],
     'post_perplexity': current_50['selected_perplexity']},
])
display(historical_comparison_df)

artifact

In [ ]:
provenance_df = pd.DataFrame([
    {'artifact': 'swiglu-3', 'path': str(SWIGLU_3_ARTIFACT_PATH.relative_to(PROJECT_ROOT)),
     'sha256': hashlib.sha256(SWIGLU_3_ARTIFACT_PATH.read_bytes()).hexdigest()},
    {'artifact': 'swiglu-2 reference', 'path': str(REFERENCE_ARTIFACT_PATH.relative_to(PROJECT_ROOT)),
     'sha256': reference_sha256},
])
display(provenance_df)
example_evaluation = results['calibration']['model_evaluation'][0]
evaluation_df = pd.DataFrame([
    {'partition': 'C4 allocation selection', 'use': 'Calibration selection / reporting',
     'batches': example_evaluation['allocation_selection']['batches'],
     'predicted_tokens': example_evaluation['allocation_selection']['predicted_tokens']},
    {'partition': 'C4 recovery validation', 'use': 'Checkpoint selection',
     'batches': configuration['data']['partition_batches']['recovery_validation'],
     'predicted_tokens': None},
    {'partition': 'WikiText validation', 'use': 'Reported language-model quality',
     'batches': example_evaluation['wikitext_validation']['batches'],
     'predicted_tokens': example_evaluation['wikitext_validation']['predicted_tokens']},
])
display(evaluation_df)
print('This notebook does not write or modify experiment artifacts.')

#### Report

The numerical findings below are derived from the loaded artifact. All local
fitting metrics are measured before global recovery. Recovery improvement does
not imply the same percentage improvement in downstream task accuracy.

In [ ]:
first_calibration = calibration_summary_df.iloc[0]
selected_calibration = calibration_summary_df.query('selected').iloc[0]
selected_block_gains = calibration_block_df.query('calibration_pairs == @SELECTED_CALIBRATION_PAIRS')
print(
    f'Calibration: selected {SELECTED_CALIBRATION_PAIRS:,} pairs per block; '
    f'WikiText PPL {first_calibration["wikitext_perplexity"]:.3f} -> '
    f'{selected_calibration["wikitext_perplexity"]:.3f} before recovery.'
)
print(
    f'Local NMSE improved in {(selected_block_gains["nmse_improvement_pct"] > 0).sum()}'
    f'/{len(selected_block_gains)} blocks relative to {baseline_pairs:,} pairs.'
)
for row in quality_summary_df.to_dict('records'):
    print(
        f'{float(row["sparsity_key"]):.0%} eligible-MLP sparsity: '
        f'{row["model_parameter_reduction_pct"]:.2f}% whole-model reduction; '
        f'WikiText PPL {row["pre_perplexity"]:.3f} -> {row["selected_perplexity"]:.3f}; '
        f'selected at {row["selected_tokens"] / 1e6:.2f}M tokens.'
    )
print(
    'Share of selected-model cross-entropy improvement reached at 10M: '
    f'{recovery_gain_df["share_of_loss_improvement_at_10m_pct"].min():.1f}% to '
    f'{recovery_gain_df["share_of_loss_improvement_at_10m_pct"].max():.1f}%.'
)

Interpretation:

- The calibration sweep measures larger data and fitting budgets jointly. Only
  the selected budget is recovered, so it does not show whether recovery would
  remove the differences between calibration budgets.
- The recovery curves show how the measured improvement develops with tokens.
  They do not establish an optimal stopping budget or justify extrapolation
  beyond the saved trajectory.
- The sparsity sweep tests one inherited policy. Uniform allocation, alternative
  temperatures, and width floors were not retested under these recovery budgets.
- Results describe one model and one run. WikiText uses the recorded short-context
  validation subset, not a full test-set or downstream benchmark evaluation.
  The dense reference is inherited from the verified swiglu-2 artifact.
- Training memory and runtime are measured for this workflow. Inference latency,
  inference memory, and deployment speedup still require separate measurements.